# Generalization: Overfitting, Regularization and Model Selection

Every model in that notebook was trained until it scored 100% on the data it had already seen, and none of them was ever shown a held-out example. This notebook is about why that number means nothing on its own.

The running example is deliberately the simplest thing that can overfit — fitting a polynomial of degree $d$ to $n$ noisy samples of a smooth curve:

$$y_i=f(x_i)+\varepsilon_i,\quad \varepsilon_i\sim\mathcal{N}(0,\sigma^2),
\qquad \hat{f}(x)=\sum_{j=0}^{d}w_j\,P_j(x)$$

Everything has a closed form, so each claim can be checked against theory rather than believed. $d$ is the knob for model capacity; $n$ and $\sigma$ set how hard the problem is. The last section swaps to classification, because the moment classes are imbalanced, accuracy becomes actively misleading.

The basis is Legendre rather than $1,x,x^2,\dots$ for a practical reason: at $d=12$ the raw monomial design matrix has condition number $8.6\times10^{9}$, and what looks like overfitting is partly floating-point collapse. In the Legendre basis the same matrix sits at $3.1\times10^{2}$, so every effect below is statistical.

In [8]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from numpy.polynomial.legendre import legvander

plt.rcParams.update({
    "figure.dpi": 108, "font.size": 9, "axes.titlesize": 9.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

CTR, CTE, CVA, CF = "#1f77b4", "#b2182b", "#7f4fbf", "#111111"
SIG = 0.22
DMAX = 12
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="320px"), "continuous_update": False}


def f_true(x):
    return np.sin(2 * np.pi * x) + 0.4 * x


def design(x, d):
    """Legendre basis on [0,1] — same span as 1,x,...,x^d but well conditioned."""
    return legvander(2 * np.asarray(x) - 1, d)


def fit(x, y, d, lam=1e-9):
    """Ridge solution; lam=1e-9 is numerical hygiene, not regularization."""
    A = design(x, d)
    I = np.eye(d + 1); I[0, 0] = 0.0          # never penalize the intercept
    return np.linalg.solve(A.T @ A + lam * I, A.T @ y)


def mse(w, x, y, d):
    return float(np.mean((design(x, d) @ w - y) ** 2))


def sample(n, sigma=SIG, seed=0):
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    return x, f_true(x) + rng.normal(0, sigma, n)


XTE, YTE = sample(4000, SIG, seed=2)          # a large honest test set
XG = np.linspace(0, 1, 400)


def curve_panel(ax, x, y, w, d, title):
    ax.plot(XG, f_true(XG), color="0.6", lw=1.6, ls="--", label="true $f$")
    ax.plot(XG, design(XG, d) @ w, color=CTE, lw=2.0, label=f"fit, d={d}")
    ax.plot(x, y, "o", ms=5, color=CTR, mec="w", mew=0.6, label="training data")
    ax.set_ylim(-2.0, 2.2); ax.set_xlim(0, 1)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.legend(fontsize=7.5, loc="upper right"); ax.set_title(title)


print(f"cond(monomial, d=12) = {np.linalg.cond(np.vander(sample(18)[0], 13)):.1e}")
print(f"cond(Legendre, d=12) = {np.linalg.cond(design(sample(18)[0], 12)):.1e}")
print(f"noise floor: sigma^2 = {SIG ** 2:.4f} — no model can beat this")

cond(monomial, d=12) = 8.6e+09
cond(Legendre, d=12) = 3.1e+02
noise floor: sigma^2 = 0.0484 — no model can beat this


## 1 — Training error always improves. That is the problem.

Increasing $d$ can only reduce training error, because a degree-$d$ polynomial contains every polynomial of lower degree. Training error is therefore not a measure of quality — it is a measure of capacity, and it reaches zero when $d+1=n$ regardless of whether anything was learned.

$$\underbrace{\text{MSE}_{\text{train}}}_{\text{monotone in }d}\quad\text{vs}\quad\underbrace{\text{MSE}_{\text{test}}}_{\text{U-shaped}}$$

With $n=18$ and $\sigma=0.22$, test error bottoms out at $d=5$ (MSE $0.0735$) and then climbs steeply: at $d=12$ it is **810× worse** than at the optimum, while training error has quietly improved from $0.5375$ to $0.0309$. That gap between the two curves is the whole subject.

Now drag `training points`. The optimum shifts right, and more importantly the *penalty* for excess capacity collapses — the median test MSE at $d=10$, measured over 25 draws, falls from 4183 at $n=12$ to 0.052 at $n=150$ — by then indistinguishable from the optimum. Overfitting is not a property of the model alone; it is a relationship between capacity and the amount of data.

| symptom | likely cause | what to try |
|---|---|---|
| train high, test high | underfitting — too little capacity | raise $d$ |
| train low, test high | overfitting | lower $d$, add data, regularize |
| train ≈ test, both high | irreducible noise, or wrong model family | check $\sigma^2$ |

In [9]:
def draw_capacity(d, n, sigma):
    x, y = sample(n, sigma, seed=1)
    w = fit(x, y, min(d, n - 1))
    ds = np.arange(0, DMAX + 1)
    tr = [mse(fit(x, y, min(k, n - 1)), x, y, min(k, n - 1)) for k in ds]
    te = [mse(fit(x, y, min(k, n - 1)), XTE, YTE, min(k, n - 1)) for k in ds]

    fig, ax = plt.subplots(1, 2, figsize=(11.4, 3.9))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.86, bottom=0.15, wspace=0.24)
    curve_panel(ax[0], x, y, w, min(d, n - 1),
                f"n = {n}, σ = {sigma:.2f}")
    ax[1].semilogy(ds, tr, "o-", color=CTR, lw=1.8, ms=4, label="training MSE")
    ax[1].semilogy(ds, te, "o-", color=CTE, lw=1.8, ms=4, label="test MSE")
    ax[1].axhline(sigma ** 2, color="0.5", lw=1.0, ls=":",
                  label="noise floor $\\sigma^2$")
    ax[1].axvline(d, color=CF, lw=1.2, ls="--")
    ax[1].plot([int(np.argmin(te))], [min(te)], "*", ms=14, color=CVA, zorder=5)
    ax[1].set_xlabel("polynomial degree d"); ax[1].set_ylabel("MSE")
    ax[1].legend(fontsize=7.5)
    ax[1].set_title(f"best d = {int(np.argmin(te))}   |   at d={d}: "
                    f"train {tr[d]:.4f}, test {te[d]:.4f}")
    plt.show()


w1 = dict(d=widgets.IntSlider(value=3, min=0, max=DMAX, step=1,
                              description="degree d:", **SL),
          n=widgets.IntSlider(value=18, min=8, max=150, step=2,
                              description="training points:", **SL),
          sigma=widgets.FloatSlider(value=SIG, min=0.02, max=0.5, step=0.02,
                                    description="noise σ:", **SL))
display(widgets.HBox([w1["d"], w1["n"], w1["sigma"]]),
        widgets.interactive_output(draw_capacity, w1))

Output()

## 2 — Where the error actually comes from

Retrain the same model on many different draws of the noise and the fits scatter. Expected test error at a point splits into exactly three pieces:

$$\mathbb{E}\big[(y-\hat{f}(x))^2\big]=
\underbrace{\big(\mathbb{E}[\hat{f}(x)]-f(x)\big)^2}_{\text{bias}^2}
+\underbrace{\mathbb{E}\big[(\hat{f}(x)-\mathbb{E}[\hat{f}(x)])^2\big]}_{\text{variance}}
+\underbrace{\sigma^2}_{\text{irreducible}}$$

The cell below estimates all three from 400 retrainings and checks the identity holds — it does, to within 0.3–2.3% at every degree. Bias falls from $0.387$ at $d=0$ to effectively zero by $d=5$ and stays there — past that point the residual wiggle is Monte-Carlo noise at the $10^{-4}$ level, not signal. Variance rises strictly monotonically at every degree, $0.0023$ to $0.0537$. Their sum is U-shaped. For least squares with an orthogonal design the variance term is predicted to be $\sigma^2(d+1)/n$, which at $d=3,n=18$ gives $0.0108$ against a measured $0.0111$.

A high-bias model is *wrong the same way every time*; a high-variance model is *wrong differently every time*. The left panel shows which one you are looking at: tight but misplaced, or scattered around the truth.

In [3]:
NREP, NFIX = 400, 18
XFIX = np.linspace(0.03, 0.97, NFIX)                # fixed design, noise resampled
YFIX = np.array([f_true(XFIX) + np.random.default_rng(700 + r).normal(0, SIG, NFIX)
                 for r in range(NREP)])
PRED = {d: np.array([design(XG, d) @ fit(XFIX, YFIX[r], d) for r in range(NREP)])
        for d in range(DMAX + 1)}
PRED_TE = {d: np.array([design(XTE, d) @ fit(XFIX, YFIX[r], d) for r in range(NREP)])
           for d in range(DMAX + 1)}
BIAS2 = {d: float(np.mean((PRED_TE[d].mean(0) - f_true(XTE)) ** 2))
         for d in range(DMAX + 1)}
VAR = {d: float(np.mean(PRED_TE[d].var(0))) for d in range(DMAX + 1)}
EMP = {d: float(np.mean([np.mean((PRED_TE[d][r] - YTE) ** 2) for r in range(NREP)]))
       for d in range(DMAX + 1)}


def draw_biasvar(d, show):
    P = PRED[d]
    fig, ax = plt.subplots(1, 2, figsize=(11.4, 3.9))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.86, bottom=0.15, wspace=0.24)

    for r in range(show):
        ax[0].plot(XG, P[r], color=CTE, lw=0.7, alpha=0.25)
    ax[0].plot(XG, P.mean(0), color=CTE, lw=2.4, label="average fit")
    ax[0].plot(XG, f_true(XG), color="0.35", lw=2.0, ls="--", label="true $f$")
    ax[0].set_ylim(-2.0, 2.2); ax[0].set_xlim(0, 1)
    ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].legend(fontsize=7.5)
    ax[0].set_title(f"{show} of {NREP} retrainings at d = {d}\n"
                    f"gap to dashed line = bias, spread = variance")

    ds = np.arange(DMAX + 1)
    b = np.array([BIAS2[k] for k in ds]); v = np.array([VAR[k] for k in ds])
    ax[1].semilogy(ds, b, "o-", color=CTR, lw=1.8, ms=4, label="bias$^2$")
    ax[1].semilogy(ds, v, "o-", color=CTE, lw=1.8, ms=4, label="variance")
    ax[1].semilogy(ds, b + v + SIG ** 2, "o-", color=CF, lw=2.0, ms=4,
                   label="sum + $\\sigma^2$")
    ax[1].semilogy(ds, [EMP[k] for k in ds], "x--", color=CVA, lw=1.4, ms=7,
                   label="measured test MSE")
    ax[1].axvline(d, color=CF, lw=1.2, ls="--")
    ax[1].set_xlabel("degree d"); ax[1].set_ylabel("error"); ax[1].legend(fontsize=7)
    err = abs(BIAS2[d] + VAR[d] + SIG ** 2 - EMP[d]) / EMP[d]
    ax[1].set_title(f"at d={d}: bias² {BIAS2[d]:.4f} + var {VAR[d]:.4f} + "
                    f"{SIG ** 2:.4f} = {BIAS2[d] + VAR[d] + SIG ** 2:.4f}"
                    f"   (measured {EMP[d]:.4f}, off by {err:.1%})")
    plt.show()


w2 = dict(d=widgets.IntSlider(value=1, min=0, max=DMAX, step=1,
                              description="degree d:", **SL),
          show=widgets.IntSlider(value=40, min=5, max=120, step=5,
                                 description="fits drawn:", **SL))
display(widgets.HBox([w2["d"], w2["show"]]),
        widgets.interactive_output(draw_biasvar, w2))

Output()

## 3 — Regularization: keep the capacity, penalize the coefficients

Rather than lowering $d$, leave it high and add a penalty that makes large coefficients expensive:

$$\hat{\mathbf{w}}=\arg\min_{\mathbf{w}}\ \|A\mathbf{w}-\mathbf{y}\|^2+\lambda\|\mathbf{w}\|^2
\qquad\Longrightarrow\qquad
\hat{\mathbf{w}}=(A^{\top}A+\lambda I)^{-1}A^{\top}\mathbf{y}$$

This is ridge regression, and in a neural network it is called **weight decay** — the same penalty, since $\frac{\partial}{\partial w}\lambda\|w\|^2=2\lambda w$ subtracts a constant fraction of every weight at each step.

At $d=12$ with $n=18$ the unregularized fit reaches test MSE $59.5$ with $\|\mathbf{w}\|=28.5$. Sweeping $\lambda$: at $10^{-3}$ test MSE is $0.454$, at $10^{-2}$ it is $0.166$, at $10^{-1}$ it is $0.102$ with $\|\mathbf{w}\|=1.60$, and it bottoms out at $\lambda=1$ with $0.095$ — a **628× improvement** from changing one number, with the degree untouched. Push on to $\lambda=10$ and it climbs back to $0.251$: too much penalty is its own bias.

$\lambda$ trades variance for bias continuously, which is why it is preferred to the discrete choice of $d$. Watch the coefficient bars collapse as you drag it.

In [10]:
LAMS = np.logspace(-9, 1.5, 44)


def draw_ridge(d, log_lam):
    lam = 10.0 ** log_lam
    x, y = sample(18, SIG, seed=1)
    w = fit(x, y, d, lam)
    tr = [mse(fit(x, y, d, l), x, y, d) for l in LAMS]
    te = [mse(fit(x, y, d, l), XTE, YTE, d) for l in LAMS]

    fig, ax = plt.subplots(1, 3, figsize=(13.0, 3.7))
    fig.subplots_adjust(left=0.055, right=0.985, top=0.84, bottom=0.16, wspace=0.28)
    curve_panel(ax[0], x, y, w, d, f"d = {d}, λ = {lam:.1e}")
    ax[1].loglog(LAMS, tr, color=CTR, lw=1.8, label="training MSE")
    ax[1].loglog(LAMS, te, color=CTE, lw=1.8, label="test MSE")
    ax[1].axvline(lam, color=CF, lw=1.2, ls="--")
    ax[1].plot([LAMS[int(np.argmin(te))]], [min(te)], "*", ms=14, color=CVA, zorder=5)
    ax[1].set_xlabel("λ"); ax[1].set_ylabel("MSE"); ax[1].legend(fontsize=7.5)
    ax[1].set_title(f"best λ = {LAMS[int(np.argmin(te))]:.1e}\n"
                    f"here: train {mse(w, x, y, d):.4f}, test {mse(w, XTE, YTE, d):.4f}")
    ax[2].bar(np.arange(d + 1), w, color=CVA)
    ax[2].axhline(0, color="0.4", lw=0.8)
    ax[2].set_xlabel("coefficient index j"); ax[2].set_ylabel("$w_j$")
    ax[2].set_ylim(-max(3.0, np.abs(w).max() * 1.1),
                   max(3.0, np.abs(w).max() * 1.1))
    ax[2].set_title(f"‖w‖ = {np.linalg.norm(w):.2f}")
    plt.show()


w3 = dict(d=widgets.IntSlider(value=12, min=1, max=DMAX, step=1,
                              description="degree d:", **SL),
          log_lam=widgets.FloatSlider(value=-9, min=-9, max=1.5, step=0.25,
                                      description="log₁₀ λ:", **SL))
display(widgets.HBox([w3["d"], w3["log_lam"]]),
        widgets.interactive_output(draw_ridge, w3))

Output()

## 4 — Early stopping: the regularizer you get for free

Fit the same over-flexible model by gradient descent from $\mathbf{w}=\mathbf{0}$ instead of solving in one step. Gradient descent moves fastest along the directions the data constrains best, so the low-frequency structure is captured within a few dozen steps and the wiggles that chase noise only arrive much later.

$$\mathbf{w}_{t+1}=\mathbf{w}_t-\eta\,\frac{2}{n}A^{\top}(A\mathbf{w}_t-\mathbf{y})$$

Training error falls monotonically forever. Validation error falls, reaches a minimum, then rises — and stopping at that minimum is a genuine regularizer. Over 60 independent noise draws at $d=12,n=18$ the median best step is **24**, and stopping there gives a test MSE **5.8× better** than running on (interquartile range 2.7× to 9.5×). The single draw plotted below is a mild one — it gains about 2.5×, and the panel reports its own numbers. That spread is worth noticing: the size of the effect depends on the noise you happened to get, which is precisely why the stopping point has to be chosen on a validation set rather than fixed in advance.

This is not a coincidence of this problem. For least squares, gradient descent stopped at step $t$ shrinks each eigendirection by roughly $1-(1-\eta\lambda_i)^t$, which is a smooth approximation to ridge's $\lambda_i/(\lambda_i+\lambda)$. **Early stopping and weight decay are approximately the same regularizer**, one applied through time and the other through the objective.

In [11]:
DGD, NGD, NSTEP = 12, 18, 3000
_xg, _yg = sample(NGD, SIG, seed=1)
_xv, _yv = sample(150, SIG, seed=3)
_A, _Av, _At = design(_xg, DGD), design(_xv, DGD), design(XTE, DGD)
_L = np.linalg.eigvalsh(_A.T @ _A * 2 / NGD).max()
_w = np.zeros(DGD + 1)
TRAJ, HIST = [], []
for _t in range(NSTEP + 1):
    TRAJ.append(_w.copy())
    HIST.append((np.mean((_A @ _w - _yg) ** 2),
                 np.mean((_Av @ _w - _yv) ** 2),
                 np.mean((_At @ _w - YTE) ** 2)))
    _w = _w - (1.0 / _L) * (_A.T @ (_A @ _w - _yg)) * 2 / NGD
HIST = np.array(HIST)
BEST = int(np.argmin(HIST[:, 1]))


def draw_earlystop(step):
    w = TRAJ[step]
    fig, ax = plt.subplots(1, 2, figsize=(11.4, 3.9))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.86, bottom=0.15, wspace=0.24)
    curve_panel(ax[0], _xg, _yg, w, DGD, f"d = {DGD} after {step} gradient steps")
    s = np.arange(NSTEP + 1)
    ax[1].semilogx(s[1:], HIST[1:, 0], color=CTR, lw=1.8, label="training")
    ax[1].semilogx(s[1:], HIST[1:, 1], color=CVA, lw=1.8, label="validation")
    ax[1].semilogx(s[1:], HIST[1:, 2], color=CTE, lw=1.4, ls=":", label="test")
    ax[1].axvline(max(step, 1), color=CF, lw=1.2, ls="--")
    ax[1].plot([max(BEST, 1)], [HIST[BEST, 1]], "*", ms=15, color=CVA, zorder=5)
    ax[1].set_xlabel("gradient step"); ax[1].set_ylabel("MSE")
    ax[1].set_ylim(0, 0.6); ax[1].legend(fontsize=7.5)
    ax[1].set_title(f"validation minimum at step {BEST} (test {HIST[BEST, 2]:.4f})\n"
                    f"at convergence test {HIST[-1, 2]:.4f} — "
                    f"{HIST[-1, 2] / HIST[BEST, 2]:.1f}× worse on this draw")
    plt.show()


w4 = dict(step=widgets.IntSlider(value=0, min=0, max=NSTEP, step=10,
                                 description="gradient step:",
                                 style={"description_width": "104px"},
                                 layout=widgets.Layout(width="560px"),
                                 continuous_update=False))
display(w4["step"], widgets.interactive_output(draw_earlystop, w4))

IntSlider(value=0, continuous_update=False, description='gradient step:', layout=Layout(width='560px'), max=30…

Output()

## 5 — Choosing $d$ and $\lambda$ without ever touching the test set

Every optimum above was read off a test curve, which is cheating: use the test set to choose and it stops being a test set. $k$-fold cross-validation solves this using only training data — split into $k$ parts, train on $k-1$, score on the one left out, rotate:

$$\text{CV}_k=\frac{1}{k}\sum_{i=1}^{k}\text{MSE}\big(\hat{f}^{(-i)},\ \text{fold}_i\big)$$

It works, with an important qualification. Across 60 independent datasets of $n=50$, the degree chosen by 5-fold CV landed within 10% of the best achievable test error **70% of the time** — good, not infallible.

The error bars show why it is not better than that. The fold-to-fold spread is around $\pm0.03$ while the entire difference between $d=3$ and $d=9$ is about $0.01$: **CV's own noise is larger than the effect it is being asked to measure.** The honest reading of a CV curve is not "the argmin is correct" but "everything from $d=3$ upward is indistinguishable, so take the simplest one." Raising $k$ shrinks the bias of each estimate but costs $k$ fits and leaves the folds more correlated.

In [6]:
def draw_cv(n, k, seed):
    x, y = sample(n, SIG, seed=700 + seed)
    idx = np.arange(n); folds = np.array_split(idx, k)
    # a fold's training split must have more points than the model has parameters
    dmax_eff = int(min(DMAX, n - max(len(f) for f in folds) - 2))
    ds = np.arange(dmax_eff + 1)
    means, sds, true = [], [], []
    for d in ds:
        e = [mse(fit(x[np.setdiff1d(idx, folds[i])], y[np.setdiff1d(idx, folds[i])],
                     d), x[folds[i]], y[folds[i]], d) for i in range(k)]
        means.append(np.mean(e)); sds.append(np.std(e))
        true.append(mse(fit(x, y, d), XTE, YTE, d))
    means, sds, true = map(np.array, (means, sds, true))
    pick = int(np.argmin(means)); best = int(np.argmin(true))

    fig, ax = plt.subplots(1, 2, figsize=(11.6, 3.9))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.15, wspace=0.26)
    ax[0].imshow(np.array([[1 if i == j else 0 for i in range(k)] for j in range(k)]),
                 cmap="Purples", aspect="auto", vmin=0, vmax=1.6)
    for j in range(k):
        ax[0].text(j, j, "val", ha="center", va="center", fontsize=8, color="w")
        for i in range(k):
            if i != j:
                ax[0].text(i, j, "train", ha="center", va="center", fontsize=7,
                           color="0.45")
    ax[0].set_xticks(range(k)); ax[0].set_yticks(range(k))
    ax[0].set_xlabel("fold"); ax[0].set_ylabel("round"); ax[0].grid(False)
    ax[0].set_title(f"{k}-fold split of n = {n} — each fold trains on "
                    f"{n - max(len(f) for f in folds)}\ndegrees above "
                    f"{dmax_eff} cannot be fitted on that many points")

    ax[1].errorbar(ds, means, yerr=sds, color=CVA, lw=1.8, marker="o", ms=4,
                   capsize=3, label=f"{k}-fold CV ± fold spread")
    ax[1].plot(ds, true, color=CTE, lw=1.6, ls="--", marker="x", ms=5,
               label="true test MSE (unavailable in practice)")
    ax[1].axhline(SIG ** 2, color="0.5", lw=1.0, ls=":")
    ax[1].set_yscale("log"); ax[1].set_xlabel("degree d"); ax[1].set_ylabel("MSE")
    ax[1].legend(fontsize=7.5)
    ax[1].set_title(f"CV picks d = {pick}, truth prefers d = {best} — "
                    f"test MSE {true[pick]:.4f} vs {true[best]:.4f} "
                    f"({true[pick] / true[best]:.2f}× off)")
    plt.show()


w5 = dict(n=widgets.IntSlider(value=50, min=20, max=200, step=10,
                              description="dataset size n:", **SL),
          k=widgets.IntSlider(value=5, min=2, max=10, step=1,
                              description="folds k:", **SL),
          seed=widgets.IntSlider(value=7, min=0, max=40, step=1,
                                 description="dataset draw:", **SL))
display(widgets.HBox([w5["n"], w5["k"], w5["seed"]]),
        widgets.interactive_output(draw_cv, w5))

Output()

## 6 — When the task is classification, accuracy lies

Switch to two classes where only 8% of examples are positive — fraud, disease screening, defect detection. A model that answers "negative" every single time scores **91.4% accuracy** while being completely useless. Accuracy is a weighted average dominated by whichever class is larger.

Count the four outcomes instead, and read two numbers off them:

$$\text{precision}=\frac{TP}{TP+FP}\ \ (\textit{when it says yes, how often is it right}),\qquad
\text{recall}=\frac{TP}{TP+FN}\ \ (\textit{of all real positives, how many were caught})$$

$$F_1=2\cdot\frac{\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}}$$

Neither is a property of the model alone — both depend on the threshold applied to its probability, and the classifier does not choose that for you. Measured on the trained model: at the default $0.5$, precision is $0.76$ but recall only $0.48$, so more than half the positives are missed while accuracy reads a comfortable $0.94$. Drop the threshold to $0.1$ and recall climbs to $0.85$ while precision falls to $0.36$ — and accuracy *drops* to $0.86$, below the do-nothing baseline.

Which is correct depends entirely on the relative cost of a missed positive versus a false alarm, and no metric can supply that for you.

In [7]:
def imbalanced(n, pos, seed):
    rng = np.random.default_rng(seed)
    y = (rng.random(n) < pos).astype(int)
    X = np.where(y[:, None] == 1, rng.normal([1.1, 1.1], 1.0, (n, 2)),
                 rng.normal([-0.4, -0.4], 1.0, (n, 2)))
    return X, y


def logistic(X, y, steps=4000, lr=0.5):
    A = np.c_[X, np.ones(len(X))]; w = np.zeros(3)
    for _ in range(steps):
        p = 1 / (1 + np.exp(-np.clip(A @ w, -60, 60)))
        w -= lr * A.T @ (p - y) / len(y)
    return w


CACHE = {}


def draw_metrics(pos_rate, thr):
    if pos_rate not in CACHE:
        Xa, ya = imbalanced(4000, pos_rate, 21)
        CACHE[pos_rate] = (logistic(Xa, ya), *imbalanced(4000, pos_rate, 22))
    w, X, y = CACHE[pos_rate]
    p = 1 / (1 + np.exp(-np.clip(np.c_[X, np.ones(len(X))] @ w, -60, 60)))
    yp = (p > thr).astype(int)
    TP = int(((yp == 1) & (y == 1)).sum()); FP = int(((yp == 1) & (y == 0)).sum())
    FN = int(((yp == 0) & (y == 1)).sum()); TN = int(((yp == 0) & (y == 0)).sum())
    prec = TP / max(TP + FP, 1); rec = TP / max(TP + FN, 1)
    f1 = 2 * prec * rec / max(prec + rec, 1e-9)

    fig, ax = plt.subplots(1, 3, figsize=(13.0, 3.8))
    fig.subplots_adjust(left=0.05, right=0.985, top=0.82, bottom=0.15, wspace=0.3)
    ax[0].scatter(*X[y == 0].T, s=5, c="0.7", label="negative")
    ax[0].scatter(*X[y == 1].T, s=9, c=CTE, label="positive")
    gx = np.linspace(X[:, 0].min(), X[:, 0].max(), 60)
    ax[0].plot(gx, -(w[0] * gx + w[2] - np.log(thr / (1 - thr))) / w[1],
               color=CF, lw=2)
    ax[0].set_ylim(X[:, 1].min(), X[:, 1].max()); ax[0].legend(fontsize=7.5)
    ax[0].set_title(f"{pos_rate:.0%} positive — boundary moves with the threshold")

    M = np.array([[TN, FP], [FN, TP]])
    ax[1].imshow(M, cmap="Blues", vmin=0, vmax=M.max())
    for (i, j), v in np.ndenumerate(M):
        ax[1].text(j, i, f"{['TN','FP','FN','TP'][i*2+j]}\n{v}", ha="center",
                   va="center", fontsize=10,
                   color="w" if v > M.max() * 0.55 else "0.1")
    ax[1].set_xticks([0, 1]); ax[1].set_xticklabels(["pred neg", "pred pos"])
    ax[1].set_yticks([0, 1]); ax[1].set_yticklabels(["true neg", "true pos"])
    ax[1].grid(False)
    ax[1].set_title(f"accuracy {(yp == y).mean():.3f}   "
                    f"(always-negative gives {1 - y.mean():.3f})")

    ths = np.linspace(0.02, 0.95, 120)
    P, R, F = [], [], []
    for t in ths:
        q = (p > t).astype(int)
        tp = ((q == 1) & (y == 1)).sum(); fp = ((q == 1) & (y == 0)).sum()
        fn = ((q == 0) & (y == 1)).sum()
        pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
        P.append(pr); R.append(rc); F.append(2 * pr * rc / max(pr + rc, 1e-9))
    ax[2].plot(ths, P, color=CTR, lw=1.8, label="precision")
    ax[2].plot(ths, R, color=CTE, lw=1.8, label="recall")
    ax[2].plot(ths, F, color=CVA, lw=1.8, label="$F_1$")
    ax[2].axvline(thr, color=CF, lw=1.2, ls="--")
    ax[2].plot([ths[int(np.argmax(F))]], [max(F)], "*", ms=14, color=CVA)
    ax[2].set_xlabel("threshold"); ax[2].set_ylim(0, 1.02); ax[2].legend(fontsize=7.5)
    ax[2].set_title(f"at {thr:.2f}: precision {prec:.2f}, recall {rec:.2f}, "
                    f"$F_1$ {f1:.2f}\nbest $F_1$ at threshold "
                    f"{ths[int(np.argmax(F))]:.2f}")
    plt.show()


w6 = dict(pos_rate=widgets.SelectionSlider(options=[0.02, 0.05, 0.08, 0.2, 0.5],
                                           value=0.08,
                                           description="positive rate:", **SL),
          thr=widgets.FloatSlider(value=0.5, min=0.02, max=0.95, step=0.02,
                                  description="threshold:", **SL))
display(widgets.HBox([w6["pos_rate"], w6["thr"]]),
        widgets.interactive_output(draw_metrics, w6))

Output()